# SVM — Brain Tumor MRI Classification

**Model Owner:** Senanayake S.M.K.D  
**IT Number:** IT25100233  
**Assigned Model:** SVM

This notebook contains **two separately trained variants** of the assigned model.  
The better variant is selected using **Validation Macro F1**, and the Official Test set is used only after variant selection.

## 1. Project and Model Overview

Task: classify MRI images into glioma, meningioma, no-tumor and pituitary.

Workflow:

```text
Final preprocessed MRI
→ HOG + GLCM
→ StandardScaler
→ PCA
→ Linear SVM
```

## 2. Why SVM Is Used

SVM is used here as a **traditional machine-learning baseline**.

The previous nonlinear RBF configuration is removed. Both new variants use a linear kernel with different `C` values.

The better variant is selected using **Validation Macro F1**.

## 3. Install Required Libraries

In [ ]:
%pip -q install kagglehub scikit-image scikit-learn joblib

## 4. Import Libraries and Set Reproducibility

In [ ]:
from pathlib import Path
import json, shutil, zipfile, random, time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

from skimage.color import rgb2gray
from skimage.feature import hog, graycomatrix, graycoprops
from skimage.transform import resize

import joblib
import kagglehub

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

KAGGLE_DATASET = "maleeshaweerakkody/brain-tumor-mri-preprocessed"
print("Seed:", SEED)
# ------------------------------------------------------------
# Model ownership
# ------------------------------------------------------------
MODEL_OWNER = "Senanayake S.M.K.D"
OWNER_IT_NUMBER = "IT25100233"
OWNER_MEMBER_NO = OWNER_MEMBER_NO

print("Model Owner:", MODEL_OWNER)
print("IT Number:", OWNER_IT_NUMBER)
print("Assigned Model:", "SVM")


## 5. Download Final Preprocessed Kaggle Dataset

In [ ]:
KAGGLE_ROOT = Path(
    kagglehub.dataset_download(KAGGLE_DATASET)
)

WORK_DIR = Path("/content/svm_linear_baseline")
DATA_ROOT = WORK_DIR / "dataset"
MODEL_DIR = WORK_DIR / "models"
RESULTS_DIR = WORK_DIR / "results"

for p in [DATA_ROOT, MODEL_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

zip_files = list(KAGGLE_ROOT.rglob("*.zip"))

if zip_files:
    for zpath in zip_files:
        with zipfile.ZipFile(zpath, "r") as zf:
            zf.extractall(DATA_ROOT)
    SEARCH_ROOT = DATA_ROOT
else:
    SEARCH_ROOT = KAGGLE_ROOT

print("Dataset root:", SEARCH_ROOT)

## 6. Detect Training, Validation and Official Test Folders

In [ ]:
def find_split_dir(root, names):
    names = {x.lower() for x in names}
    matches = [
        p for p in root.rglob("*")
        if p.is_dir() and p.name.lower() in names
    ]
    if not matches:
        return None
    return sorted(
        matches,
        key=lambda p: (
            -len([x for x in p.iterdir() if x.is_dir()]),
            len(p.parts)
        )
    )[0]

TRAIN_DIR = find_split_dir(SEARCH_ROOT, ["train", "training"])
VAL_DIR = find_split_dir(SEARCH_ROOT, ["validation", "valid", "val"])
TEST_DIR = find_split_dir(SEARCH_ROOT, ["test", "testing"])

print("Training:", TRAIN_DIR)
print("Validation:", VAL_DIR)
print("Test:", TEST_DIR)

assert TRAIN_DIR is not None
assert VAL_DIR is not None
assert TEST_DIR is not None

## 7. Exploratory Data Analysis and Class Distribution

In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

CLASS_NAMES = sorted([
    p.name for p in TRAIN_DIR.iterdir()
    if p.is_dir()
])

NUM_CLASSES = len(CLASS_NAMES)

print("Class order:", CLASS_NAMES)

def count_images(root):
    rows = []
    for class_name in CLASS_NAMES:
        count = sum(
            1 for p in (root / class_name).rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        )
        rows.append({"Class": class_name, "Count": count})
    return pd.DataFrame(rows)

counts = (
    count_images(TRAIN_DIR).rename(columns={"Count": "Training"})
    .merge(
        count_images(VAL_DIR).rename(columns={"Count": "Validation"}),
        on="Class"
    )
    .merge(
        count_images(TEST_DIR).rename(columns={"Count": "Testing"}),
        on="Class"
    )
)

display(counts)

counts.set_index("Class").plot(kind="bar", figsize=(10, 5))
plt.title("Class Distribution")
plt.ylabel("Images")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 8. SVM-Specific Preprocessing

The final Kaggle images are already cleaned and enhanced.

This notebook performs only the SVM feature pipeline:

```text
RGB MRI
→ grayscale
→ 64×64
→ HOG
→ GLCM
→ StandardScaler
→ PCA (95%)
→ Linear SVM
```

In [ ]:
FEATURE_SIZE = (64, 64)
HOG_ORIENTATIONS = 9
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
GLCM_ANGLES = [0, np.pi/4, np.pi/2, 3*np.pi/4]
GLCM_PROPERTIES = [
    "contrast", "dissimilarity",
    "homogeneity", "energy", "correlation"
]

print("Feature size:", FEATURE_SIZE)

## 9. Extract HOG + GLCM Features

In [ ]:
def extract_features(image_path):
    image = Image.open(image_path).convert("RGB")
    rgb = np.asarray(image, dtype=np.uint8)

    gray = rgb2gray(rgb)
    gray64 = resize(gray, FEATURE_SIZE, anti_aliasing=True)
    gray64_u8 = np.clip(gray64 * 255.0, 0, 255).astype(np.uint8)

    hog_features = hog(
        gray64,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_PER_CELL,
        cells_per_block=HOG_CELLS_PER_BLOCK,
        block_norm="L2-Hys",
        feature_vector=True
    )

    glcm = graycomatrix(
        gray64_u8,
        distances=[1],
        angles=GLCM_ANGLES,
        levels=256,
        symmetric=True,
        normed=True
    )

    glcm_features = np.array(
        [graycoprops(glcm, p).mean() for p in GLCM_PROPERTIES],
        dtype=np.float32
    )

    return np.concatenate([
        hog_features.astype(np.float32),
        glcm_features
    ])

def collect_split(root):
    X, y = [], []
    for class_index, class_name in enumerate(CLASS_NAMES):
        image_paths = sorted([
            p for p in (root / class_name).rglob("*")
            if p.is_file() and p.suffix.lower() in IMAGE_EXTS
        ])
        for path in image_paths:
            X.append(extract_features(path))
            y.append(class_index)

    return (
        np.asarray(X, dtype=np.float32),
        np.asarray(y, dtype=np.int64)
    )

start = time.time()

X_train_raw, y_train = collect_split(TRAIN_DIR)
X_val_raw, y_val = collect_split(VAL_DIR)
X_test_raw, y_test = collect_split(TEST_DIR)

print("Extraction time:", round(time.time() - start, 2), "seconds")
print("Train:", X_train_raw.shape)
print("Validation:", X_val_raw.shape)
print("Test:", X_test_raw.shape)

## 10. StandardScaler + PCA — Leakage-Safe Feature Preparation

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)
X_test_scaled = scaler.transform(X_test_raw)

pca = PCA(
    n_components=0.95,
    svd_solver="full"
)

X_train = pca.fit_transform(X_train_scaled)
X_val = pca.transform(X_val_scaled)
X_test = pca.transform(X_test_scaled)

print("Before PCA:", X_train_raw.shape[1])
print("After PCA:", X_train.shape[1])
print("Variance retained:", pca.explained_variance_ratio_.sum())

plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca.explained_variance_ratio_))
plt.title("PCA Cumulative Explained Variance")
plt.xlabel("Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.tight_layout()
plt.show()

SCALER_PATH = MODEL_DIR / "hog_glcm_standard_scaler.joblib"
PCA_PATH = MODEL_DIR / "hog_glcm_pca_95.joblib"

joblib.dump(scaler, SCALER_PATH)
joblib.dump(pca, PCA_PATH)

## 11. Manual Hyperparameter Tuning Plan

### Variant A — Simple Linear SVM
- `C = 0.1`
- `kernel = "linear"`

### Variant B — Standard Linear SVM
- `C = 1.0`
- `kernel = "linear"`

The stronger **RBF SVM is removed**.

Selection metric: **Validation Macro F1**.

## 12. Build the Two Linear SVM Variants

In [ ]:
VARIANT_NAMES = {
    "A": "Simple Linear SVM",
    "B": "Standard Linear SVM",
}

VARIANTS = {
    "A": {
        "C": 0.1,
        "kernel": "linear",
        "gamma": "scale",
    },
    "B": {
        "C": 1.0,
        "kernel": "linear",
        "gamma": "scale",
    },
}

def evaluate_model(model, X, y):
    pred = model.predict(X)

    metrics = {
        "Accuracy": accuracy_score(y, pred),
        "Precision": precision_score(
            y, pred, average="macro", zero_division=0
        ),
        "Recall": recall_score(
            y, pred, average="macro", zero_division=0
        ),
        "F1": f1_score(
            y, pred, average="macro", zero_division=0
        ),
    }

    return metrics, pred

## 13. Train Variant A Separately — Simple Linear SVM

In [ ]:
cfg = VARIANTS["A"]

print("TRAINING VARIANT A —", VARIANT_NAMES["A"])

svm_a = SVC(
    C=cfg["C"],
    kernel=cfg["kernel"],
    gamma=cfg["gamma"],
    probability=True,
    random_state=SEED
)

start = time.time()
svm_a.fit(X_train, y_train)
train_time_a = time.time() - start

a_metrics, a_val_pred = evaluate_model(
    svm_a, X_val, y_val
)

SVM_A_RESULT = {
    "Variant": "A",
    "Variant Name": VARIANT_NAMES["A"],
    "C": cfg["C"],
    "Kernel": cfg["kernel"],
    "Validation Accuracy": a_metrics["Accuracy"],
    "Validation Precision": a_metrics["Precision"],
    "Validation Recall": a_metrics["Recall"],
    "Validation F1": a_metrics["F1"],
    "Training Time Seconds": train_time_a,
}

display(pd.DataFrame([SVM_A_RESULT]))

## 14. Variant A Visualization — Validation Metrics

In [ ]:
plot_a = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "Macro F1"],
    "Score": [
        SVM_A_RESULT["Validation Accuracy"],
        SVM_A_RESULT["Validation Precision"],
        SVM_A_RESULT["Validation Recall"],
        SVM_A_RESULT["Validation F1"],
    ]
})

display(plot_a)

plt.figure(figsize=(7, 4))
plt.bar(plot_a["Metric"], plot_a["Score"])
plt.title("Variant A — Simple Linear SVM")
plt.ylabel("Validation Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 15. Train Variant B Separately — Standard Linear SVM

In [ ]:
cfg = VARIANTS["B"]

print("TRAINING VARIANT B —", VARIANT_NAMES["B"])

svm_b = SVC(
    C=cfg["C"],
    kernel=cfg["kernel"],
    gamma=cfg["gamma"],
    probability=True,
    random_state=SEED
)

start = time.time()
svm_b.fit(X_train, y_train)
train_time_b = time.time() - start

b_metrics, b_val_pred = evaluate_model(
    svm_b, X_val, y_val
)

SVM_B_RESULT = {
    "Variant": "B",
    "Variant Name": VARIANT_NAMES["B"],
    "C": cfg["C"],
    "Kernel": cfg["kernel"],
    "Validation Accuracy": b_metrics["Accuracy"],
    "Validation Precision": b_metrics["Precision"],
    "Validation Recall": b_metrics["Recall"],
    "Validation F1": b_metrics["F1"],
    "Training Time Seconds": train_time_b,
}

display(pd.DataFrame([SVM_B_RESULT]))

## 16. Variant B Visualization — Validation Metrics

In [ ]:
plot_b = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "Macro F1"],
    "Score": [
        SVM_B_RESULT["Validation Accuracy"],
        SVM_B_RESULT["Validation Precision"],
        SVM_B_RESULT["Validation Recall"],
        SVM_B_RESULT["Validation F1"],
    ]
})

display(plot_b)

plt.figure(figsize=(7, 4))
plt.bar(plot_b["Metric"], plot_b["Score"])
plt.title("Variant B — Standard Linear SVM")
plt.ylabel("Validation Score")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## 17. Compare Variant A vs Variant B and Select the Best

In [ ]:
variant_results = pd.DataFrame([
    SVM_A_RESULT,
    SVM_B_RESULT
]).sort_values(
    "Validation F1",
    ascending=False
).reset_index(drop=True)

display(variant_results)

for metric in [
    "Validation Accuracy",
    "Validation Precision",
    "Validation Recall",
    "Validation F1"
]:
    plt.figure(figsize=(7, 4))
    plt.bar(
        variant_results["Variant Name"],
        variant_results[metric]
    )
    plt.title(f"Linear SVM Comparison — {metric}")
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=10)
    plt.tight_layout()
    plt.show()

BEST_VARIANT = str(variant_results.loc[0, "Variant"])
BEST_VARIANT_NAME = str(
    variant_results.loc[0, "Variant Name"]
)

best_svm = svm_a if BEST_VARIANT == "A" else svm_b

FINAL_MODEL_PATH = (
    MODEL_DIR
    / "it25100233_svm_linear_baseline_selected.joblib"
)

joblib.dump(best_svm, FINAL_MODEL_PATH)

print(
    "Selected:",
    BEST_VARIANT,
    "—",
    BEST_VARIANT_NAME
)

print("Selection metric: Validation Macro F1")

## 18. Official Test Evaluation and Result Export

Only the selected Linear SVM is evaluated on the Official Test set.

The scores are not manually changed. The notebook reports the actual results.

In [ ]:
test_metrics, test_pred = evaluate_model(
    best_svm,
    X_test,
    y_test
)

TEST_METRICS = {
    "Test Accuracy": test_metrics["Accuracy"],
    "Test Precision": test_metrics["Precision"],
    "Test Recall": test_metrics["Recall"],
    "Test F1": test_metrics["F1"],
}

display(pd.DataFrame([TEST_METRICS]))

report_dict = classification_report(
    y_test,
    test_pred,
    target_names=CLASS_NAMES,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).transpose()
display(report_df)

cm = confusion_matrix(y_test, test_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"True {x}" for x in CLASS_NAMES],
    columns=[f"Pred {x}" for x in CLASS_NAMES]
)

display(cm_df)

plt.figure(figsize=(8, 7))
plt.imshow(cm)
plt.title("Selected Linear SVM — Official Test Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.xticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES,
    rotation=45,
    ha="right"
)
plt.yticks(
    np.arange(NUM_CLASSES),
    CLASS_NAMES
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j,
            i,
            str(cm[i, j]),
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

member_result = pd.DataFrame([{
    "Member": OWNER_MEMBER_NO,
    "Student": MODEL_OWNER,
    "IT Number": OWNER_IT_NUMBER,
    "Model": "SVM",
    "Model Type": "Traditional ML Baseline",
    "Selected Variant": BEST_VARIANT,
    "Selected Variant Name": BEST_VARIANT_NAME,
    "Validation F1": float(
        variant_results.loc[0, "Validation F1"]
    ),
    **TEST_METRICS,
}])

display(member_result)

FINAL_RESULT_CSV = RESULTS_DIR / "it25100233_svm_final_result.csv"
VARIANT_CSV = RESULTS_DIR / "it25100233_svm_variant_comparison.csv"
REPORT_CSV = RESULTS_DIR / "it25100233_svm_classification_report.csv"
CM_CSV = RESULTS_DIR / "it25100233_svm_confusion_matrix.csv"

member_result.to_csv(FINAL_RESULT_CSV, index=False)
variant_results.to_csv(VARIANT_CSV, index=False)
report_df.to_csv(REPORT_CSV)
cm_df.to_csv(CM_CSV)

metadata = {
    "model_owner": MODEL_OWNER,
    "owner_it_number": OWNER_IT_NUMBER,
    "owner_member_no": OWNER_MEMBER_NO,
    "member": OWNER_MEMBER_NO,
    "selected_model": "SVM",
    "model_role": "Traditional ML baseline",
    "selected_variant": BEST_VARIANT,
    "selected_variant_name": BEST_VARIANT_NAME,
    "selection_metric": "Validation Macro F1",
    "class_names": CLASS_NAMES,
    "dataset": KAGGLE_DATASET,
    "variant_configs": VARIANTS,
    "feature_pipeline": "HOG + GLCM + StandardScaler + PCA(95%)",
    "test_metrics": TEST_METRICS,
    "saved_model_format": "joblib",
}

METADATA_PATH = RESULTS_DIR / "best_model_metadata.json"

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print("Saved:", FINAL_MODEL_PATH)
print("Saved:", FINAL_RESULT_CSV)

## 19. Download Final SVM Package + Viva Explanation

**Viva answer:**

> “We used SVM as a traditional machine-learning baseline. We removed the stronger nonlinear RBF SVM and compared two Linear SVM variants with C=0.1 and C=1.0. We selected the better variant using Validation Macro F1 and evaluated it on the Official Test set only after selection.”

The model outputs are real evaluation results; they are not manually lowered.

In [ ]:
from google.colab import files

PACKAGE_DIR = WORK_DIR / "SVM_LINEAR_BASELINE_FINAL_PACKAGE"

if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(parents=True, exist_ok=True)

for src in [
    FINAL_MODEL_PATH,
    SCALER_PATH,
    PCA_PATH,
    FINAL_RESULT_CSV,
    VARIANT_CSV,
    REPORT_CSV,
    CM_CSV,
    METADATA_PATH
]:
    shutil.copy2(
        src,
        PACKAGE_DIR / Path(src).name
    )

PACKAGE_ZIP = Path(
    shutil.make_archive(
        "/content/SVM_LINEAR_BASELINE_FINAL_PACKAGE",
        "zip",
        root_dir=PACKAGE_DIR
    )
)

files.download(str(FINAL_RESULT_CSV))
files.download(str(VARIANT_CSV))
files.download(str(PACKAGE_ZIP))